### Intialize

In [1]:
from google.colab import drive
import os

if not os.path.exists("/content/drive"):
    print("Drive not mounted. Mounting now...")
    drive.mount('/content/drive')
else:
    print("Drive already mounted")


Drive already mounted


In [2]:
!pip install traci
!pip install stable_baselines3
!pip install colorama
!pip install optuna

### Pathes and parameters


#### Modifiable

In [1]:
path_project_folder = "/content/drive/My Drive/study/graduation_project/final/Code/project_files/TrafficManager/TrafficManager"


#### Fixed


In [2]:
import sys
import yaml
sys.path.append(path_project_folder)
sys.path.insert(0, path_project_folder)


from Callbacks import *
from models.d3qn import D3QNAgent
from data_parser import *
from Observations.sumo_obs import DefaultObservation
import SumoEnvSingleAgent
from Utils_reporting import *
from Utils_running_singleAgent import *
from rewards import *
from Connections import SumoConnection
from Connections.Connection import *
from Observations.db_obs import DBObservationProto
from Connections.DBTrainingConnection import ONLINEDBConn



Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
data_path = path_project_folder + "/DBOnline/Data/simple_data.db"
db_path = path_project_folder+"/DBOnline/model_estimaitor.keras"
yaml_file = path_project_folder + "/config.yaml"
keys_file = path_project_folder + "/keys.env"
path_info = path_project_folder + "/info_road.csv"
log_file = "sumo_log.txt"


In [4]:
TABLE_NAME = "Stefano_STATE"
ACTION_LENGTH = 8 # as in db


In [5]:
reward_func = {
    'proposed_reward': reward_proposed,
    'literature_reward': reward_liter,
    'project_reward': reward_proj,
}

env_classes = {
    "HighGroupedSumoEnv": SumoEnvSingleAgent.HighGroupedSumoEnv,
    "GroupedSumoEnv": SumoEnvSingleAgent.GroupedSumoEnv,
    "SumoEnv": SumoEnvSingleAgent.SumoEnv,
}

with open(yaml_file, "r") as file:
    config = yaml.safe_load(file)

general_settings = config['general_settings']
experiment_settings_changable = config["experiment_settings"]['changable_settings']
experiment_settings_const = config["experiment_settings"]["const_settings"]
algorithm_settings = config["algorithms_settings"]

# General settings
is_gui = general_settings['is_gui']
see_progress_each = general_settings['see_progress_each']
enable_variation_action = general_settings["enable_variation_action"]
yellow_time = general_settings["yellow_time"]

# Constant experiment settings
max_steps = experiment_settings_const["max_steps"]
n_env = experiment_settings_const["n_env"]
durations = experiment_settings_const["durations"]
enable_gcd = experiment_settings_const["enable_gcd"]
n_episode_evaluation = experiment_settings_const["n_episode_evaluation"]
larger_evaluation = experiment_settings_const["larger_evaluation"]

enable_gcd = experiment_settings_const["enable_gcd"]
n_episode_evaluation = experiment_settings_const["n_episode_evaluation"]
larger_evaluation = experiment_settings_const["larger_evaluation"]

if enable_gcd:
    step_size, reduced_durations = gcd_and_reduced(durations)
else:
    step_size = 1
    reduced_durations = durations


In [6]:
# Changable experiment settings
data_name = 'Stefano' # the parameter only affect when not optimizing
ENV_NAME = 'HighGroupedSumoEnv'
REWARD_TYPE = 'proposed'
seed = 0
# Data folder paths
if data_name == 'Mosheer':
    path_data_folder = path_project_folder + "AIST/data2_mosheerIsmail/"
else:
    path_data_folder = path_project_folder + "AIST/data3_san_stefano/"

path_cfg = path_data_folder + "cfg.sumocfg"





In [7]:
def create_env(config_):
    """Create the simulation environment with given config."""
    env = env_classes[ENV_NAME](
        data_name=data_name,
        durations=reduced_durations,
        reward_fun=reward_func[REWARD_TYPE+"_reward"],
        step_size=step_size,
        obs_class=DefaultObservation,
        path_info=path_info,
        yellow_time=yellow_time,
        max_steps=max_steps,
        sumo_traffic_scale=.2, # not necessary
        enable_variation_action=enable_variation_action,
        config=config_,
        seed=seed
    )
    env.data_path = path_data_folder
    env.see_progress_each = see_progress_each
    return env

### Database functions

In [8]:
import sys
sys.path.append(path_project_folder+"/DBOnline")
from db_functions import *

In [9]:

## Create database and Insert random data

#create_database(data_path,TABLE_NAME)
#insert_sample_data(TABLE_NAME,data_path,30,ACTION_LENGTH)

### Model

In [10]:
epochs=100
batch_size=2

In [11]:
import ast
import numpy as np

data = load_data(TABLE_NAME,data_path)

states_before = []
actions = []
states_after = []

for i in range(len(data)):
    states_before.append(np.array(ast.literal_eval(data[i][0])))
    actions.append(np.array(data[i][1]))
    states_after.append(np.array(ast.literal_eval(data[i][2])))

states_before=np.array(states_before)
actions=np.array(actions)
states_after=np.array(states_after)


In [12]:
from tensorflow import keras
from tensorflow.keras import layers


if not os.path.exists(db_path):
    state_dim = len(states_before[0])

    state_input = keras.Input(shape=(state_dim,), name="state_before")
    action_input = keras.Input(shape=(1,), name="action")
    concatenated = layers.concatenate([state_input, action_input])

    x = layers.Dense(64, activation="relu")(concatenated)
    x = layers.Dense(64, activation="relu")(x)
    output = layers.Dense(state_dim, name="state_after")(x)


    model_estimaitor = keras.Model(inputs=[state_input, action_input], outputs=output)

    # Compile the model
    model_estimaitor.compile(optimizer="adam", loss="mse")
    model_estimaitor.summary()

In [13]:
if not os.path.exists(db_path):
    model_estimaitor.fit([states_before, actions], states_after, epochs=epochs, batch_size=batch_size)

In [14]:
if os.path.exists(db_path):
  model_estimaitor = keras.models.load_model(db_path)

### Test

In [15]:
conn = ONLINEDBConn(len(states_before))

In [16]:
set_global_conn(conn)

In [17]:
obs = DBObservationProto(conn,states_before, actions, states_after,model_estimaitor)

In [18]:
env = create_env({})

In [19]:
env.observations = {key:obs for key,val in env.observations.items()}

In [20]:
env.reset()
conn.reset()

Environment reset - agents properly initialized.


(array([0., 0., 0., 0., 0., 0., 0.], dtype=float32), {})

In [21]:
env.encoded_action_mapping

{0: ('rrrr', 17),
 1: ('rrrr', 30),
 2: ('rrrr', 60),
 3: ('rrrr', 90),
 4: ('gggg', 17),
 5: ('gggg', 30),
 6: ('gggg', 60),
 7: ('gggg', 90)}

In [22]:
action = env.action_space.sample()
print(action, "-->", env.encoded_action_mapping[action])
before = conn.getTime()
print("---> Before ",before)
env.step(action)
after = conn.getTime()
print("---> After ",after)
print("Difference ",(after-before) )

4 --> ('gggg', 17)
---> Before  0
state_before_i [ 59.49   6.47 250.42  18.98  47.06  35.72  51.96]
action_ 4
model_prediction [[ 34.197273    2.6782713 210.89253    25.500841   35.043667   21.063354
   57.887417 ]]
state_before_i [ 59.49   6.47 250.42  18.98  47.06  35.72  51.96]
action_ 4
model_prediction [[ 34.197273    2.6782713 210.89253    25.500841   35.043667   21.063354
   57.887417 ]]
---> After  17
Difference  17


In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

# Wrap for SB3
vec_env = DummyVecEnv([lambda: env])

# Train PPO model
ppo_model = PPO("MlpPolicy", vec_env, verbose=1)
ppo_model.learn(total_timesteps=5000)


# Save
ppo_model.save("ppo_sqlite_model")


In [24]:
obs = vec_env.reset()
for _ in range(10):
    action, _ = ppo_model.predict(obs)
    obs, reward, done, info = vec_env.step(action)
    print(f"Action: {action}, Reward: {reward}")
    if done:
        obs = vec_env.reset()


Environment reset - agents properly initialized.
state_before_i [ 35.23   6.49 277.21  13.22  34.42   2.23  53.6 ]
action_ 1
model_prediction [[ 6.7165017  4.8421416 42.99435   10.663954  12.236705  31.785755
  29.074709 ]]
state_before_i [ 35.23   6.49 277.21  13.22  34.42   2.23  53.6 ]
action_ 1
model_prediction [[ 6.7165017  4.8421416 42.99435   10.663954  12.236705  31.785755
  29.074709 ]]
Action: [1], Reward: [-0.28852594]
state_before_i [ 46.4    9.78 299.3   64.71  11.7   13.3   12.06]
action_ 1
model_prediction [[ 11.013854   4.110788 177.4911    23.473251  31.56707   29.265175
   25.088247]]
state_before_i [ 46.4    9.78 299.3   64.71  11.7   13.3   12.06]
action_ 1
model_prediction [[ 11.013854   4.110788 177.4911    23.473251  31.56707   29.265175
   25.088247]]
Action: [1], Reward: [-0.00953127]
state_before_i [ 16.24   4.55 164.33  34.64  15.14  25.61  40.48]
action_ 6
model_prediction [[ 27.330303    3.7245336 102.14258    33.992477   23.62145    21.352087
   39.581055 

### Bug : It may add yellow durartion here even if color is not changed buetween phases